# Goal

Аннилинг `learn_rate` на **3-м шаге** (шаг пластичный) вместо 5-го. Parent `07_old:57` (чемпион `42.2`), `ent_coef=const(0.05)`.
Проверяем: даёт ли форма кривой (warmup→спад) что-то сверх ровного `const`.

# TARGET_NOTEBOOK_FNAME


In [13]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07_old.ipynb'

# OPTUNA_DECISIVE_METRIC


In [14]:
OPTUNA_DECISIVE_METRIC = 'test/levels_passed_mean'

# GRID_SEARCH_SPACE


In [15]:
# @launchit.collect
GRID_SEARCH_SPACE = dict(
    schedule=[
        'const_lo',
        'const_hi',
        'linear',
        'shark_fin',
    ],
    random_seed=list(range(1, 111, 10)),  # 11 seeds -> grid 4x11=44, launches_count=40 => 10 per arm
)

# set_hyperparameters


In [16]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    HP.general.random_seed = optuna_trial.suggest_categorical('random_seed', GRID_SEARCH_SPACE['random_seed'])
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True

    HP.env.count = 32
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    wm_parent = '18d_world_model_09:183'
    HP.vision_head.parent = dict(model=wm_parent, weights=wm_parent)
    HP.vision_head.is_trainable = False

    agent_parent = '18n_ppo_tr_frostbite_07_old:57'
    HP.encoder.parent = dict(model=wm_parent, weights=agent_parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = agent_parent
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10
    HP.agent.d_model = 256
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True

    HP.test.env_rams = None
    HP.test.env_ram_patches = None
    HP.test.break_on_level_passed = False

    # Rollout: keep nine_lives (deep chains) + natural play.
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = [
        ['nine_lives'],
    ]

    # Training procedure params (PPO related)
    HP.ppo.global_steps_count = 3_000_000
    HP.ppo.rollout_steps_count = 512
    HP.ppo.epochs_count = 2
    HP.ppo.batch_size = 512
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.vf_coef = 0.2
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1

    schedule = optuna_trial.suggest_categorical('schedule', GRID_SEARCH_SPACE['schedule'])

    lr_hi, lr_lo = '0.00025', '0.00008'
    
    if schedule == 'const_lo':
        HP.ppo.learn_rate = f'const({lr_lo})'
    elif schedule == 'const_hi':
        HP.ppo.learn_rate = f'const({lr_hi})'
    elif schedule == 'linear':
        HP.ppo.learn_rate = f'linear({lr_hi}, {lr_lo})'
    elif schedule == 'shark_fin':
        HP.ppo.learn_rate = f'shark_fin({lr_hi}, {lr_lo}, 0.05)'
    else:
        assert False, schedule

    HP.ppo.ent_coef = 'const(0.05)'

    HP.ppo.gamma = 0.997
    HP.ppo.gae_lambda = 0.95
    HP.ppo.clip_coef = 0.1
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5
    HP.ppo.target_kl = None
    HP.ppo.norm_adv = True

    return HP

# Results

**На пластичном 3-м шаге эффект ЕСТЬ — и правит УРОВЕНЬ lr, а не форма кривой.** Низкий lr (`const(8e-5)`) и оба аннилинга `hi→lo` дают ~2.9–3.0, а ровный высокий `const(2.5e-4)` — только 2.35 (ANOVA p=0.005, η²=0.30). Между низким `const` и аннилингами разницы нет ⇒ аннилинг сводится к «меньше времени на высоком lr».

Ось = форма `learn_rate`; parent жёстко `07_old:57`, `lr_hi=2.5e-4`, `lr_lo=8e-5`, `ent=const(0.05)`, cold-тест, rollout `[nine_lives]`. `random_seed` из сетки → парный дизайн. Применение расписаний проверено по TB `anneal/a) learn_rate`: **0/40 несоответствий**.

| рука | расписание | n | Mean | Best (модель) | Std |
|---|---|---|---|---|---|
| `const_lo` | `const(8e-5)` | 9 | **3.028** | 3.844 (`:296`) | 0.410 |
| `linear` | `linear(2.5e-4, 8e-5)` | 11 | **3.031** | 3.875 (`:302`) | 0.512 |
| `shark_fin` | `shark_fin(2.5e-4, 8e-5, 0.05)` | 9 | 2.879 | 3.000 (`:299`) | 0.101 |
| `const_hi` | `const(2.5e-4)` | 11 | **2.349** | 3.438 (`:290`) | 0.627 |
| — | ALL | 40 | 2.809 | 3.875 (`:302`) | 0.539 |

ANOVA: **F=5.03, p=0.0052, η²=0.295**. Парно по сидам vs `const_hi`: `const_lo` Δ=+0.767 (p=0.028), `linear` +0.682 (p=0.013), `shark_fin` +0.556 (p=0.035). Парно vs `const_lo`: `linear` Δ=+0.007 (p=0.97), `shark_fin` Δ=−0.023 (p=0.80) — **различий нет**.

Корреляции (n=40): исход vs финальный lr r=−0.53 (p=0.0004), vs средний lr r=−0.47 (p=0.002), vs доля шагов на high-lr r=−0.43 (p=0.005); vs стартовый lr н/з (r=−0.22). Решает «сколько времени на высоком lr», не старт.

vs baseline `42.3` (mean 2.447): `42.3b` ALL 2.809, Welch p=0.010 (d=+0.70); `const_hi` ≡ рецепт `42.3` → p=0.66 (согласованная реплика). Прирост дают низко-lr руки, не форма.

Системная проблема: **не проверено по журналу**.
Данные: журнал пережил recovery 2026-10-17 (было 35 версий → 40; пропуск версий 320,321); руки 9/11/11/9 (не 10/10/10/10).

<img src="./img/42_3b_summary.png">

**Выводы**
1) Форма кривой `learn_rate` **не работает** и на 3-м шаге: аннилинг `hi→lo` статистически равен ровному low, обе бьют ровный high. Работает **уровень** lr.
2) На 5-м шаге этот же вывод дал null по уровню (агент заморожен, +1% подвижности); на 3-м уровень виден (η²=30%). Гипотеза «деревянеет к 5-му» подтверждается.
3) Следующий шаг: низкий ровный lr как рабочий дефолт; вместо аннилинга — фокус на curriculum/пластичности.

# System

In [17]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [18]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07_old.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07_old',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_42.3b/18n_study_42.3b.ipynb',
 'optuna_study_name': '18n_study_42.3b',
 'optuna_study_serial': '42.3b',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_42.3b/18n_study_42.3b.optuna'}

In [19]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [20]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [21]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [22]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [28]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 5
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-17 22:21:35,129] Using an existing study with name '18n_study_42.3b' instead of creating a new one.


2026.10.17-22:21:35.460100     0.308 >> Model instance registered, version=322
2026.10.17-22:21:35.474282     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch322.ipynb"
2026.10.17-22:21:35.474878     0.006 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07_old:322"; running launches=1
2026.10.17-22:22:06.476350     0.294 >> Model instance registered, version=323
2026.10.17-22:22:06.488088     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch323.ipynb"
2026.10.17-22:22:06.488354     0.004 >> 5.7 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07_old:323"; running launches=2
2026.10.17-22:22:37.542600     0.279 >> Model instance registered, version=324
2026.10.17-22:22:37.554809     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch324.ipynb"
2026.10.17-22:22:37.555095     0.005 >> 4.5 idle runners exist, submitted launch "18n

In [ ]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

In [ ]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

In [ ]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()